In [3]:
import os

import torch
import esm
import esm.esmfold.v1.esmfold as esmfold_module
from omegaconf import OmegaConf
import gc
import pandas as pd

In [4]:
TRUNK_PATH = r"C:\Users\admin\.cache\torch\hub\checkpoints\esmfold_3B_v1.pt"
LM_PATH = r"C:\Users\admin\.cache\torch\hub\checkpoints\esm2_t36_3B_UR50D.pt"

In [5]:
# ================= 1. 强制清理内存 =================
# 删除旧模型防止显存溢出
if 'model' in globals():
    del model
torch.cuda.empty_cache()
gc.collect()
print("🧹 内存已清理，准备重新加载...")

# ================= 2. 重新加载模型 (强制 FP32) =================
print(f"🔄 正在重新从磁盘加载模型 (确保是 FP32)...")

# 加载骨架
trunk_data = torch.load(TRUNK_PATH, map_location="cpu")
cfg = OmegaConf.create(trunk_data["cfg"])
model = esmfold_module.ESMFold(esmfold_config=cfg.model, num_layers=36, embed_dim=2560, attention_heads=40, token_dropout=True)

# 修复骨架键名
trunk_state = trunk_data["model"]
new_trunk = {}
for k, v in trunk_state.items():
    k = k.replace("ipa.linear_q_points.weight", "ipa.linear_q_points.linear.weight")
    k = k.replace("ipa.linear_q_points.bias", "ipa.linear_q_points.linear.bias")
    k = k.replace("ipa.linear_kv_points.weight", "ipa.linear_kv_points.linear.weight")
    k = k.replace("ipa.linear_kv_points.bias", "ipa.linear_kv_points.linear.bias")
    new_trunk[k] = v
model.load_state_dict(new_trunk, strict=False)

# 加载大脑
lm_data = torch.load(LM_PATH, map_location="cpu")
new_lm = {}
for k, v in lm_data["model"].items():
    k = k.replace("encoder.sentence_encoder.", "").replace("sentence_encoder.", "")
    new_lm[f"esm.{k}"] = v
model.load_state_dict(new_lm, strict=False)

# 关键：放入 GPU 并保持 float() (FP32)
model = model.eval().cuda().float() 
print("✅ 模型已重置为全精度 (FP32)。")

🧹 内存已清理，准备重新加载...
🔄 正在重新从磁盘加载模型 (确保是 FP32)...


C:\Users\admin\AppData\Local\Temp\ipykernel_3560\1583662298.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trunk_data = torch.load(TRUNK_PATH, map_location="cpu")
C:\U

✅ 模型已重置为全精度 (FP32)。


In [34]:
expertment_data = r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\PF00201_taxonomy_Uniprot_Level2(condition_generation_standard_aa)+expertments+SMILES_ID.csv'
format_data = pd.read_csv(expertment_data)

In [40]:
format_data_ = format_data[format_data['organism'].isin(['Stevia rebaudiana?A','Stevia rebaudiana?D'])]
format_data_

,Unnamed: 0,uniprot_id,entry_name,is_reviewed,organism,organism_id,length,seq,gene_name,data_quality_level,...,lost_loop,H,H1,H2,acceptor_id,donor_id,get_loop_id,lost_loop_id,H_id,seq_id
5575,5606,A17G,NaN,NaN,Stevia rebaudiana?A,NaN,442,MDNQNGRISILLLPFLGHGHISPFFELAKQLAKRNCNVYLCSTPIN...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00086,SMILES_00054,SMILES_00087,SMILES_00058,SMILES_00089,seq_5575
5576,5607,A17S,NaN,NaN,Stevia rebaudiana?A,NaN,442,MDNQNGRISILLLPFLSHGHISPFFELAKQLAKRNCNVYLCSTPIN...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00086,SMILES_00054,SMILES_00087,SMILES_00058,SMILES_00089,seq_5576
5577,5608,D177A,NaN,NaN,Stevia rebaudiana?A,NaN,442,MDNQNGRISILLLPFLAHGHISPFFELAKQLAKRNCNVYLCSTPIN...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00086,SMILES_00054,SMILES_00087,SMILES_00058,SMILES_00089,seq_5577
5578,5609,D177E,NaN,NaN,Stevia rebaudiana?A,NaN,442,MDNQNGRISILLLPFLAHGHISPFFELAKQLAKRNCNVYLCSTPIN...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00086,SMILES_00054,SMILES_00087,SMILES_00058,SMILES_00089,seq_5578
5579,5610,D177F,NaN,NaN,Stevia rebaudiana?A,NaN,442,MDNQNGRISILLLPFLAHGHISPFFELAKQLAKRNCNVYLCSTPIN...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00086,SMILES_00054,SMILES_00087,SMILES_00058,SMILES_00089,seq_5579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5764,5787,Valid_RebD_to_RebM_8,NaN,NaN,Stevia rebaudiana?D,NaN,442,MENKTETTVRRRRRIILFPVPFQGHINPILQLANVLYSKGFSITIF...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00087,SMILES_00054,SMILES_00088,SMILES_00058,SMILES_00089,seq_5764
5765,5787,Valid_RebD_to_RebM_9,NaN,NaN,Stevia rebaudiana?D,NaN,442,MENKTETTVRRRRRIILFPVPFQGHINPILQLANVLYSKGFSITIF...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00087,SMILES_00054,SMILES_00088,SMILES_00058,SMILES_00089,seq_5765
5766,5787,Valid_RebD_to_RebM_10,NaN,NaN,Stevia rebaudiana?D,NaN,442,MENKTETTVRRRRRIILFPVPFQGHINPILQLANVLYSKGFSITIF...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00087,SMILES_00054,SMILES_00088,SMILES_00058,SMILES_00089,seq_5766
5767,5787,Valid_RebD_to_RebM_11,NaN,NaN,Stevia rebaudiana?D,NaN,442,MENKTETTVRRRRRIILFPVPFQGHINPILQLANVLYSKGFSITIF...,NaN,3,...,O[C@@H]1[C@@H](COP([O-])(=O)OP([O-])([O-])=O)O...,[H+],NaN,NaN,SMILES_00087,SMILES_00054,SMILES_00088,SMILES_00058,SMILES_00089,seq_5767


In [26]:
# 设置分块以防 OOM
model.trunk.set_chunk_size(128)



# ================= 3. 执行推理 =================
sequence = "MDNQNGRISILLLPFLAHGHISPFFELAKQLAKRNCNVYLCSTPINLSSIKDKDPSASIKLVELHLPSSPDLPPHYHTTNGLPSHLMLPLRNAFETAGPTFSEILKTLKPDLLIYDFNPSWAPEIASSHNIPAVYFLTTAAASSSIGLHAFKNPGEKYPFPDFYDNSNITPEPPSADNMKLLHDFIACFERSCDIILIKSFRELEGKYIDLLSTLSDKTLVPVGPLVQDPMGHNEDPKTEQIINWLDKREESTVVFVCFGSEYFLSNEELEEVAIGLELSTVNFIWAVRLIEGEKKGILPEGFLQRVGDRGLVVEGWAPQARILGHSSIGGFVSHCGWSSIAESMKFGVPVIAMARHLDQPLNGKLAAEVGVGMEVVRDENGKYKREGIAEVIRKVVVEKSGEVIRRKARELSEKMKEKGEQEIDRALEELVQICKKKKDEQ"
output_pdb_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Sturecture_AF2_ESMFold\Valid_M8_ESMFold.pdb"

print(f"🧬 开始推理 (序列长度: {len(sequence)})...")
try:
    with torch.no_grad():
        output = model.infer([sequence])
        pdb_string = model.output_to_pdb(output)[0]

    with open(output_pdb_path, "w") as f:
        f.write(pdb_string)
    
    print(f"🎉 成功！文件已保存: {output_pdb_path}")

except RuntimeError as e:
    if "out of memory" in str(e):
        print("❌ 显存不足 (OOM)。尝试减小 chunk_size 为 64。")
    else:
        print(f"❌ 错误: {e}")
except IndexError as e:
    print(f"❌ 数值错误: {e}")
    print("这通常是因为使用了 FP16。确认上面是否已经强制执行了 .float()。")

🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Sturecture_AF2_ESMFold\Valid_M8_ESMFold.pdb


In [42]:
model.trunk.set_chunk_size(128)
for i in range(format_data_.shape[0]):
    row = format_data_.iloc[i]
    sequence = row['seq']
    seq_id = row['uniprot_id']
    output_pdb_path = rf"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\{seq_id}_ESMFold.pdb"
    torch.cuda.empty_cache()
    gc.collect()
    print(f"🧬 开始推理 (序列长度: {len(sequence)})...")
    try:
        with torch.no_grad():
            output = model.infer([sequence])
            pdb_string = model.output_to_pdb(output)[0]
    
        with open(output_pdb_path, "w") as f:
            f.write(pdb_string)
        
        print(f"🎉 成功！文件已保存: {output_pdb_path}")
    
    except RuntimeError as e:
        if "out of memory" in str(e):
            print("❌ 显存不足 (OOM)。尝试减小 chunk_size 为 64。")
        else:
            print(f"❌ 错误: {e}")
    except IndexError as e:
        print(f"❌ 数值错误: {e}")
        print("这通常是因为使用了 FP16。确认上面是否已经强制执行了 .float()。")
    

🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\A17G_ESMFold.pdb
🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\A17S_ESMFold.pdb
🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\D177A_ESMFold.pdb
🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\D177E_ESMFold.pdb
🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\D177F_ESMFold.pdb
🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\D177G_ESMFold.pdb
🧬 开始推理 (序列长度: 442)...
🎉 成功！文件已保存: E:\pUGTdb\20251128_collect

In [41]:
format_data_.shape

(194, 31)

# Batch Inference

In [6]:
file_list = os.listdir(r'F:\Reaction_Model')

In [7]:
def read_fasta(fasta_path):
    """
    读取 FASTA，返回 [(seq_id, seq_str), ...]
    - 支持多行序列
    - seq_id 取 '>' 后第一段（空格前）
    """
    data_list = []
    seq_id = None
    seq_chunks = []
    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                # flush previous
                if seq_id is not None:
                    seq = "".join(seq_chunks).replace(" ", "").replace("\t", "")
                    data_list.append((seq_id, seq))
                # new header
                header = line[1:].strip()
                seq_id = header.split()[0] if header else f"seq_{len(data_list)}"
                seq_chunks = []
            else:
                seq_chunks.append(line)
        # last
        if seq_id is not None:
            seq = "".join(seq_chunks).replace(" ", "").replace("\t", "")
            data_list.append((seq_id, seq))
    return data_list

In [ ]:
file_list = os.listdir(r'F:\Reaction_Model')
for file in file_list:
    file_path = os.path.join(r'F:\Reaction_Model', file)
    save_path = os.path.join(file_path, 'gent_seq\ESMFold')
    # os.mkdir(save_path)
    input_path = os.path.join(file_path, 'gent_seq\generated_samples_string.fasta')
    fasta_list = read_fasta(input_path)
    model.trunk.set_chunk_size(128)
    for seq_id,sequence in fasta_list:
        output_pdb_path = rf"{save_path}\{seq_id}_ESMFold.pdb"
        print(f"🧬 开始推理 (序列长度: {output_pdb_path})...")
        torch.cuda.empty_cache()
        gc.collect()
        print(f"🧬 开始推理 (序列长度: {len(sequence)})...")
        try:
            with torch.no_grad():
                output = model.infer([sequence])
                pdb_string = model.output_to_pdb(output)[0]

            with open(output_pdb_path, "w") as f:
                f.write(pdb_string)

            print(f"🎉 成功！文件已保存: {output_pdb_path}")

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("❌ 显存不足 (OOM)。尝试减小 chunk_size 为 64。")
            else:
                print(f"❌ 错误: {e}")
        except IndexError as e:
            print(f"❌ 数值错误: {e}")
            print("这通常是因为使用了 FP16。确认上面是否已经强制执行了 .float()。")
    

🧬 开始推理 (序列长度: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_0_ESMFold.pdb)...
🧬 开始推理 (序列长度: 491)...
🎉 成功！文件已保存: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_0_ESMFold.pdb
🧬 开始推理 (序列长度: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_1_ESMFold.pdb)...
🧬 开始推理 (序列长度: 524)...
🎉 成功！文件已保存: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_1_ESMFold.pdb
🧬 开始推理 (序列长度: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_2_ESMFold.pdb)...
🧬 开始推理 (序列长度: 366)...
🎉 成功！文件已保存: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_2_ESMFold.pdb
🧬 开始推理 (序列长度: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_3_ESMFold.pdb)...
🧬 开始推理 (序列长度: 483)...
🎉 成功！文件已保存: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_3_ESMFold.pdb
🧬 开始推理 (序列长度: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_4_ESMFold.pdb)...
🧬 开始推理 (序列长度: 481)...
🎉 成功！文件已保存: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_4_ESMFold.pdb
🧬 开始推理 (序列长度: F:\Reaction_Model\deepchem\gent_seq\ESMFold\SEQUENCE_5_E